### Reasoning (추론)

GPT-5와 같은 추론 모델은 강화 학습을 통해 추론을 수행하도록 훈련된 LLM입니다. 추론 모델은 사용자에게 응답하기 전에 먼저 생각하여 , 사용자에게 응답하기 전에 긴 내부 사고 사슬을 생성합니다. 추론 모델은 복잡한 문제 해결, 코딩, 과학적 추론, 그리고 에이전트 워크플로우를 위한 다단계 계획 수립에 탁월합니다. 또한 , 경량 코딩 에이전트인 Codex CLI 에 가장 적합한 모델입니다.

추론 모델은 입력 및 출력 토큰 외에도 추론 토큰을 도입합니다 . 모델은 이러한 추론 토큰을 사용하여 "사고"하고, 제시된 질문을 분석하며, 다양한 접근 방식을 고려하여 최적의 응답을 생성합니다. 추론 토큰을 생성한 후, 모델은 완성 토큰 형태로 응답을 출력하고, 추론 토큰은 컨텍스트에서 제거합니다.

다음은 사용자와 어시스턴트 간의 여러 단계로 이루어진 대화 예시입니다. 각 단계의 입력 및 출력 토큰은 다음 단계로 전달되지만, 추론 토큰은 버려집니다.

In [1]:
from dotenv import load_dotenv
load_dotenv() # read local .env file

True

In [2]:
from openai import OpenAI
import json

client = OpenAI()

Model = "gpt-5"

In [3]:
prompt = """
문자열로 표현된 행렬을 '[1,2],[3,4],[5,6]' 형식으로 받아서 같은 형식으로 전치행렬을 출력하는 파이썬 스크립트를 작성하세요.
"""

response = client.responses.create(
    model=Model,  
    reasoning={"effort": "medium"},  # 추론 수준 설정: 중간 수준의 정답 생성 요구
    input=[
        {
            "role": "user", 
            "content": prompt  # 사용자의 질문 또는 지시
        }
    ]
)

# 생성된 파이썬 스크립트 응답 출력
print(response.output_text)

다음은 '[1,2],[3,4],[5,6]' 형식의 문자열을 입력받아 같은 형식으로 전치행렬을 출력하는 파이썬 스크립트입니다.

#!/usr/bin/env python3
import sys, ast

def parse_matrix(s):
    try:
        data = ast.literal_eval("[" + s + "]")
    except (SyntaxError, ValueError):
        raise ValueError("입력 형식이 올바르지 않습니다. 예: [1,2],[3,4]")
    if not isinstance(data, list) or not data:
        raise ValueError("행렬이 비어있습니다.")
    rows = []
    for row in data:
        if not isinstance(row, (list, tuple)):
            raise ValueError("각 행은 대괄호로 묶인 리스트여야 합니다. 예: [1,2]")
        rows.append(list(row))
    n = len(rows[0])
    if any(len(r) != n for r in rows):
        raise ValueError("모든 행의 길이가 같아야 합니다.")
    return rows

def transpose(mat):
    if not mat or not mat[0]:
        return []
    return [list(col) for col in zip(*mat)]

def format_matrix(mat):
    return ",".join("[" + ",".join(str(v) for v in row) + "]" for row in mat)

def main():
    if len(sys.argv) > 1:
        s = " ".join(sys.argv[1:]).strip()
    else:
        s = sys.s

In [4]:
response.output

[ResponseReasoningItem(id='rs_074ea72dca4a6a360069af9256a1448194bdb8e0ea0c0a1fa1', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseOutputMessage(id='msg_074ea72dca4a6a360069af92a5fda081949fd160ad35ef2b20', content=[ResponseOutputText(annotations=[], text='다음은 \'[1,2],[3,4],[5,6]\' 형식의 문자열을 입력받아 같은 형식으로 전치행렬을 출력하는 파이썬 스크립트입니다.\n\n#!/usr/bin/env python3\nimport sys, ast\n\ndef parse_matrix(s):\n    try:\n        data = ast.literal_eval("[" + s + "]")\n    except (SyntaxError, ValueError):\n        raise ValueError("입력 형식이 올바르지 않습니다. 예: [1,2],[3,4]")\n    if not isinstance(data, list) or not data:\n        raise ValueError("행렬이 비어있습니다.")\n    rows = []\n    for row in data:\n        if not isinstance(row, (list, tuple)):\n            raise ValueError("각 행은 대괄호로 묶인 리스트여야 합니다. 예: [1,2]")\n        rows.append(list(row))\n    n = len(rows[0])\n    if any(len(r) != n for r in rows):\n        raise ValueError("모든 행의 길이가 같아야 합니다.")\n    return rows\n\nd

### Python 프로젝트 계획 및 생성

In [5]:
prompt = """
사용자 질문을 받아 답변과 매핑된 데이터베이스에서 검색하는 Python 앱을 만들고 싶습니다.
유사한 답변이 있으면 일치하는 답변을 검색합니다.
만약 일치하지 않으면 사용자에게 답변을 입력하도록 요청하고
질문/답변 쌍을 데이터베이스에 저장합니다. 필요한 디렉터리 구조를 계획한 후
각 파일을 전체적으로 반환합니다.
코드 전체가 아닌 시작과 끝에만 해설을 제공하세요.
한국어로 답하세요.
"""

response = client.responses.create(
    model=Model,
    input=[
        {
            "role": "user",
            "content": prompt,
        }
    ]
)

print(response.output_text)

아래는 간단한 CLI 기반 Python 앱의 설계와 전체 코드입니다. 동작 요약:
- SQLite 데이터베이스에 질문/답변(Q/A) 쌍을 저장합니다.
- 사용자가 질문을 입력하면 저장된 질문들과의 유사도를 계산해 가장 비슷한 질문을 찾습니다.
- 임계치 이상이면 해당 답변을 반환합니다.
- 임계치 미만이면 사용자에게 답변을 입력받아 Q/A를 저장합니다.
- 외부 라이브러리 없이 표준 라이브러리만 사용합니다.

디렉터리 구조
- app.py
- README.md
- requirements.txt
- db/
  - __init__.py
  - database.py
  - schema.sql
- services/
  - __init__.py
  - qa_service.py
  - similarity.py
- utils/
  - __init__.py
  - text.py
- data/ (런타임에 DB 파일 생성)

파일 전체

파일: app.py
```
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys
import signal
from db.database import Database
from services.qa_service import QAService

def _graceful_exit(signum, frame):
    print("\n종료합니다.")
    sys.exit(0)

def main():
    signal.signal(signal.SIGINT, _graceful_exit)
    signal.signal(signal.SIGTERM, _graceful_exit)

    db = Database(db_path="data/qa.sqlite")
    db.init_db()

    service = QAService(db=db, similarity_threshold=0.82)

    print("Q/A 매핑 CLI입니다. 질문을 입력하세요. (종료: q 또는 빈 줄)")
    while 

In [6]:
response.output

[ResponseReasoningItem(id='rs_002fddaf9f80500d0069af92ca13e88194aecb3ee87937821a', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseOutputMessage(id='msg_002fddaf9f80500d0069af92f42d7481948d1be52004c45422', content=[ResponseOutputText(annotations=[], text='아래는 간단한 CLI 기반 Python 앱의 설계와 전체 코드입니다. 동작 요약:\n- SQLite 데이터베이스에 질문/답변(Q/A) 쌍을 저장합니다.\n- 사용자가 질문을 입력하면 저장된 질문들과의 유사도를 계산해 가장 비슷한 질문을 찾습니다.\n- 임계치 이상이면 해당 답변을 반환합니다.\n- 임계치 미만이면 사용자에게 답변을 입력받아 Q/A를 저장합니다.\n- 외부 라이브러리 없이 표준 라이브러리만 사용합니다.\n\n디렉터리 구조\n- app.py\n- README.md\n- requirements.txt\n- db/\n  - __init__.py\n  - database.py\n  - schema.sql\n- services/\n  - __init__.py\n  - qa_service.py\n  - similarity.py\n- utils/\n  - __init__.py\n  - text.py\n- data/ (런타임에 DB 파일 생성)\n\n파일 전체\n\n파일: app.py\n```\n#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\nimport sys\nimport signal\nfrom db.database import Database\nfrom services.qa_service import QAService\n\ndef _graceful_exit(signum, frame):\n

## 추론 기반 코드 자동 생성 실습

**"숫자로 이루어진 문자열이 주어질 때, 해당 숫자의 평균과 중앙값을 구하는 Python 함수를 작성하라."**
예를 들어 `"1, 5, 3, 8, 2"`라는 문자열이 입력되면 평균은 `3.8`, 중앙값은 `3`입니다.


* 추론 모델을 사용하여 위 문제를 해결하는 **Python 코드**를 생성하세요.
* `reasoning={"effort": "medium"}` 또는 `reasoning={"effort": "high"}` 옵션을 사용하여 **모델의 내부 추론 과정**을 유도하세요.
* 코드 실행 결과가 **정확한지 직접 검증**해보세요.

### 구현 조건

* 문자열을 숫자 리스트로 변환하는 전처리 로직 포함
* 평균(average)과 중앙값(median)을 계산하는 함수 구성
* 코드 내에 주석 또는 단계별 설명이 포함되도록 모델에게 유도
* 출력값 확인을 위한 예제 입력 포함 (`"1, 5, 3, 8, 2"`)

* reasoning effort를 `low`, `medium`, `high`로 바꿔가며 결과물 비교
* `json.loads()`로 실행 가능한 함수 형태로 모델이 응답하도록 유도해보기